## Data Loading

In [1]:
import pandas as pd

file_path = '/content/uber_trips.parquet'
df = pd.read_parquet(file_path)
df.head()

,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,tolls,tips,driver_pay,PUBorough,PUZone,DOBorough,DOZone,time_from_request_to_pickup,day_of_week,hour_bucket
0,2026-01-01 00:50:37,2026-01-01 00:52:31,2026-01-01 00:54:30,2026-01-01 01:13:23,262,79,4.30,1133,0.0,0.00,21.10,Manhattan,Yorkville East,Manhattan,East Village,114.0,Thursday,0
1,2026-01-01 00:09:12,2026-01-01 00:12:17,2026-01-01 00:12:39,2026-01-01 00:22:42,195,52,1.89,604,0.0,0.00,9.10,Brooklyn,Red Hook,Brooklyn,Cobble Hill,185.0,Thursday,0
2,2026-01-01 00:16:16,2026-01-01 00:29:33,2026-01-01 00:31:34,2026-01-01 00:55:21,25,181,1.84,1427,0.0,2.67,21.94,Brooklyn,Boerum Hill,Brooklyn,Park Slope,797.0,Thursday,0
3,2026-01-01 00:29:54,2026-01-01 00:36:04,2026-01-01 00:36:32,2026-01-01 01:02:28,162,113,2.84,1556,0.0,0.00,20.90,Manhattan,Midtown East,Manhattan,Greenwich Village North,370.0,Thursday,0
4,2026-01-01 00:07:33,2026-01-01 00:10:16,2026-01-01 00:11:14,2026-01-01 00:14:06,22,22,0.66,172,0.0,0.00,4.31,Brooklyn,Bensonhurst West,Brooklyn,Bensonhurst West,163.0,Thursday,0


## Average Time For Zone To Zone (Per Hour Per Day)

In [2]:
grouped_data = df.groupby(['PUZone', 'DOZone', 'hour_bucket', 'day_of_week'])
average_trip_time_by_route_time = grouped_data['trip_time'].mean().reset_index()

# Calculate the count of instances for each combination
trip_count_by_route_time = grouped_data.size().reset_index(name='trip_count')

# Merge the count into the average trip time DataFrame
average_trip_time_by_route_time = pd.merge(
    average_trip_time_by_route_time,
    trip_count_by_route_time,
    on=['PUZone', 'DOZone', 'hour_bucket', 'day_of_week'],
    how='left'
)

# Set trip_time to 0 where PUZone and DOZone are the same
average_trip_time_by_route_time.loc[average_trip_time_by_route_time['PUZone'] == average_trip_time_by_route_time['DOZone'], 'trip_time'] = 0

# Rename the 'trip_time' column to 'average_trip_time'
average_trip_time_by_route_time = average_trip_time_by_route_time.rename(columns={'trip_time': 'average_PU_to_DO_time'})

print(average_trip_time_by_route_time.head())
print(average_trip_time_by_route_time.shape)

                    PUZone                   DOZone  hour_bucket day_of_week  \
0  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0      Friday   
1  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0      Monday   
2  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0    Saturday   
3  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0      Sunday   
4  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0    Thursday   

   average_PU_to_DO_time  trip_count  
0                    0.0          12  
1                    0.0          11  
2                    0.0          18  
3                    0.0          13  
4                    0.0          11  
(3138392, 6)


In [3]:
# Merge 'average_PU_to_DO_time' back into the original DataFrame 'df'
df = pd.merge(
    df,
    average_trip_time_by_route_time[['PUZone', 'DOZone', 'hour_bucket', 'day_of_week', 'average_PU_to_DO_time']],
    on=['PUZone', 'DOZone', 'hour_bucket', 'day_of_week'],
    how='left'
)

print(df.head())
print(df.shape)

# Check for missing values after the merge
print("\nMissing values after merge:")
print(df.isnull().sum())

# Print 5 random rows with specified columns
print("\n5 random rows with relevant columns:")
print(df[['PUZone', 'DOZone', 'day_of_week', 'hour_bucket', 'average_PU_to_DO_time']].sample(5))

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:29:54 2026-01-01 00:36:04 2026-01-01 00:36:32   
4 2026-01-01 00:07:33 2026-01-01 00:10:16 2026-01-01 00:11:14   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  \
0 2026-01-01 01:13:23           262            79        4.30       1133   
1 2026-01-01 00:22:42           195            52        1.89        604   
2 2026-01-01 00:55:21            25           181        1.84       1427   
3 2026-01-01 01:02:28           162           113        2.84       1556   
4 2026-01-01 00:14:06            22            22        0.66        172   

   tolls  tips  driver_pay  PUBorough            PUZone  DOBorough  \
0    0.0  0.00       21.10  Manhattan    Yorkville East  Manhattan   
1    0.0  0.

## Average Earnings For Pickup Zone (Per Hour Per Day)

In [4]:
import numpy as np

# Create actual calendar date so we don't group across many weeks/months
df['pickup_date'] = df['pickup_datetime'].dt.date

# -----------------------------
# Stage 1: daily zone-hour level
# -----------------------------
daily_zone_hour = df.groupby(
    ['PUZone', 'pickup_date', 'day_of_week', 'hour_bucket']
).agg(
    earliest_pickup_time=('pickup_datetime', 'min'),
    latest_dropoff_time=('dropoff_datetime', 'max'),
    total_driver_pay=('driver_pay', 'sum'),
    total_tips=('tips', 'sum'),
    trip_count=('pickup_datetime', 'count')
).reset_index()

daily_zone_hour['total_earnings'] = (
    daily_zone_hour['total_driver_pay'] + daily_zone_hour['total_tips']
)

daily_zone_hour['time_span_hours'] = (
    daily_zone_hour['latest_dropoff_time'] - daily_zone_hour['earliest_pickup_time']
).dt.total_seconds() / 3600

# Avoid bad/inflated values from tiny spans
daily_zone_hour = daily_zone_hour[daily_zone_hour['time_span_hours'] >= 0.10].copy()

daily_zone_hour['PU_driver_pay_per_hour'] = (
    daily_zone_hour['total_driver_pay'] / daily_zone_hour['time_span_hours']
)

daily_zone_hour['PU_tips_per_hour'] = (
    daily_zone_hour['total_tips'] / daily_zone_hour['time_span_hours']
)

daily_zone_hour['PU_total_earnings_per_hour'] = (
    daily_zone_hour['total_earnings'] / daily_zone_hour['time_span_hours']
)

# ------------------------------------------------
# Stage 2: average across same zone/day/hour pattern
# ------------------------------------------------
grouped_earnings = daily_zone_hour.groupby(
    ['PUZone', 'day_of_week', 'hour_bucket']
).agg(
    PU_avg_market_pay_per_hour=('PU_driver_pay_per_hour', 'mean'),
    PU_avg_market_tips_per_hour=('PU_tips_per_hour', 'mean'),
    PU_avg_market_total_earnings_per_hour=('PU_total_earnings_per_hour', 'mean'),
    avg_trip_count=('trip_count', 'mean'),
    avg_market_total_earnings=('total_earnings', 'mean'),
    avg_time_span_hours=('time_span_hours', 'mean'),
    sample_days=('pickup_date', 'nunique')
).reset_index()

print(grouped_earnings.head())
print(grouped_earnings.shape)

                    PUZone day_of_week  hour_bucket  \
0  Allerton/Pelham Gardens      Friday            0   
1  Allerton/Pelham Gardens      Friday            1   
2  Allerton/Pelham Gardens      Friday            2   
3  Allerton/Pelham Gardens      Friday            3   
4  Allerton/Pelham Gardens      Friday            4   

   PU_avg_market_pay_per_hour  PU_avg_market_tips_per_hour  \
0                  168.720444                     2.500901   
1                  134.133291                     0.288376   
2                   73.766992                     2.435331   
3                  111.190888                     2.607120   
4                  174.600358                     6.225114   

   PU_avg_market_total_earnings_per_hour  avg_trip_count  \
0                             171.221346       14.333333   
1                             134.421667       11.333333   
2                              76.202324        3.888889   
3                             113.798008        5.666667

In [5]:
top_earnings_rows = grouped_earnings.sort_values(by='avg_market_total_earnings', ascending=False)
print(top_earnings_rows.head())

               PUZone day_of_week  hour_bucket  PU_avg_market_pay_per_hour  \
25470  Midtown Center   Wednesday           21                 9415.803542   
25446  Midtown Center     Tuesday           21                 8805.930431   
25469  Midtown Center   Wednesday           20                 7743.176833   
25422  Midtown Center    Thursday           21                 7764.138911   
25445  Midtown Center     Tuesday           20                 7207.968859   

       PU_avg_market_tips_per_hour  PU_avg_market_total_earnings_per_hour  \
25470                   867.029439                           10282.832980   
25446                   828.203561                            9634.133992   
25469                   672.796941                            8415.973774   
25422                   695.820579                            8459.959490   
25445                   678.322154                            7886.291013   

       avg_trip_count  avg_market_total_earnings  avg_time_span_hour

In [6]:
import numpy as np

# How frequently rides occur.
grouped_earnings['PU_trip_density_per_hour'] = (
    grouped_earnings['avg_trip_count'] /
    grouped_earnings['avg_time_span_hours']
)

# Trip Quality:
grouped_earnings['PU_avg_earnings_per_trip'] = (
    grouped_earnings['avg_market_total_earnings'] /
    grouped_earnings['avg_trip_count']
)

# Higher tipping neighborhoods.
grouped_earnings['PU_tip_ratio_per_trip'] = (
    grouped_earnings['PU_avg_market_tips_per_hour'] /
    grouped_earnings['PU_avg_market_total_earnings_per_hour']
)

grouped_earnings['PU_zone_opportunity_score'] = (
    grouped_earnings['PU_avg_market_total_earnings_per_hour']
    * np.log1p(grouped_earnings['PU_trip_density_per_hour'])
)

print(grouped_earnings.sample(5))

                                 PUZone day_of_week  hour_bucket  \
10908  East Concourse/Concourse Village    Saturday           22   
34511                  South Ozone Park   Wednesday           22   
36090                         Sunnyside    Saturday           17   
14510                          Flatiron   Wednesday            0   
7718                       Clinton Hill    Saturday           21   

       PU_avg_market_pay_per_hour  PU_avg_market_tips_per_hour  \
10908                 1227.579789                    11.173138   
34511                  706.726191                    14.394316   
36090                 1700.019104                    87.329819   
14510                  533.145259                    34.108229   
7718                  2052.569504                   115.988669   

       PU_avg_market_total_earnings_per_hour  avg_trip_count  \
10908                            1238.752927      133.111111   
34511                             721.120507       80.000000   
36

**Reasoning**:
The subtask requires calculating the mean of two columns from `grouped_earnings` and then dividing them to find a conversion factor. This can be done in a single code cell.



In [7]:
avg_pu_opportunity_score = grouped_earnings['PU_zone_opportunity_score'].mean()
avg_market_earnings = grouped_earnings['avg_market_total_earnings'].mean()

conversion_factor = avg_pu_opportunity_score / avg_market_earnings

print(f"Average PU Zone Opportunity Score: {avg_pu_opportunity_score}")
print(f"Average Market Total Earnings: {avg_market_earnings}")
print(f"Conversion Factor (Opportunity Score Index Units per Dollar): {conversion_factor}")

Average PU Zone Opportunity Score: 2828.870946205133
Average Market Total Earnings: 1226.2774065861172
Conversion Factor (Opportunity Score Index Units per Dollar): 2.3068768379909566


In [8]:
columns_to_merge = [
    'PUZone',
    'day_of_week',
    'hour_bucket',
    'PU_zone_opportunity_score',
    'PU_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip',
    'PU_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour'
]

df = pd.merge(
    df,
    grouped_earnings[columns_to_merge],
    on=['PUZone', 'day_of_week', 'hour_bucket'],
    how='left'
)

print(df.head())
print(df.shape)

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:29:54 2026-01-01 00:36:04 2026-01-01 00:36:32   
4 2026-01-01 00:07:33 2026-01-01 00:10:16 2026-01-01 00:11:14   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  \
0 2026-01-01 01:13:23           262            79        4.30       1133   
1 2026-01-01 00:22:42           195            52        1.89        604   
2 2026-01-01 00:55:21            25           181        1.84       1427   
3 2026-01-01 01:02:28           162           113        2.84       1556   
4 2026-01-01 00:14:06            22            22        0.66        172   

   tolls  tips  ...  time_from_request_to_pickup day_of_week hour_bucket  \
0    0.0  0.00  ...                        114.0    Thursday           0   


In [9]:
# Check for missing values after the merge
print("\nMissing values after merge:")
print(df.isnull().sum())


Missing values after merge:
request_datetime                          0
on_scene_datetime                         0
pickup_datetime                           0
dropoff_datetime                          0
PULocationID                              0
DOLocationID                              0
trip_miles                                0
trip_time                                 0
tolls                                     0
tips                                      0
driver_pay                                0
PUBorough                                 0
PUZone                                    0
DOBorough                                 0
DOZone                                    0
time_from_request_to_pickup               0
day_of_week                               0
hour_bucket                               0
average_PU_to_DO_time                     0
pickup_date                               0
PU_zone_opportunity_score                43
PU_tip_ratio_per_trip                    47
PU_

In [10]:
import pandas as pd

missing_tip_ratio_rows = df[df['PU_tip_ratio_per_trip'].isnull()]
print("Rows with missing PU_tip_ratio_per_trip:")

# Display all columns for the first 10 rows
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(missing_tip_ratio_rows.head(10))

Rows with missing PU_tip_ratio_per_trip:
           request_datetime   on_scene_datetime     pickup_datetime    dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  tolls  tips  driver_pay      PUBorough                               PUZone      DOBorough                               DOZone  time_from_request_to_pickup day_of_week  hour_bucket  average_PU_to_DO_time pickup_date  PU_zone_opportunity_score  PU_tip_ratio_per_trip  PU_avg_earnings_per_trip  PU_trip_density_per_hour  PU_avg_market_total_earnings_per_hour
211648  2026-01-01 10:32:23 2026-01-01 10:34:39 2026-01-01 10:34:43 2026-01-01 10:40:39            99             5        1.46        356    0.0   3.0        5.72  Staten Island                      Freshkills Park  Staten Island                        Arden Heights                        136.0    Thursday           10                  356.0  2026-01-01                        NaN                    NaN                       NaN                       NaN   

In [11]:
columns_with_missing = [
    'PU_zone_opportunity_score',
    'PU_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip',
    'PU_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour'
]

original_rows = df.shape[0]
df.dropna(subset=columns_with_missing, inplace=True)

print(f"Dropped {original_rows - df.shape[0]} rows with missing values.")
print(f"New DataFrame shape: {df.shape}")
print("Remaining missing values after dropping:")
print(df[columns_with_missing].isnull().sum())

Dropped 47 rows with missing values.
New DataFrame shape: (23474050, 25)
Remaining missing values after dropping:
PU_zone_opportunity_score                0
PU_tip_ratio_per_trip                    0
PU_avg_earnings_per_trip                 0
PU_trip_density_per_hour                 0
PU_avg_market_total_earnings_per_hour    0
dtype: int64


In [12]:
columns_to_merge_from_grouped_earnings = [
    'PUZone',
    'day_of_week',
    'hour_bucket',
    'PU_zone_opportunity_score',
    'PU_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip',
    'PU_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour'
]

# Create a temporary DataFrame for merging DOZone specific features
do_zone_metrics = grouped_earnings[columns_to_merge_from_grouped_earnings].copy()

# Rename PUZone to DOZone for merging on drop-off zone
do_zone_metrics.rename(columns={'PUZone': 'DOZone'}, inplace=True)

# Rename PU_ prefixed columns to DO_ prefixed columns
do_zone_metrics.rename(columns={
    'PU_zone_opportunity_score': 'DO_zone_opportunity_score',
    'PU_tip_ratio_per_trip': 'DO_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip': 'DO_avg_earnings_per_trip',
    'PU_trip_density_per_hour': 'DO_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour': 'DO_avg_market_total_earnings_per_hour'
}, inplace=True)

# Merge these new DOZone specific features into the main DataFrame
df = pd.merge(
    df,
    do_zone_metrics,
    on=['DOZone', 'day_of_week', 'hour_bucket'],
    how='left'
)

# Calculate the opportunity cost of relocation time
# The opportunity cost is the earnings lost in the PUZone during the relocation time.
# average_PU_to_DO_time is in seconds, so convert to hours by dividing by 3600.
relocation_opportunity_cost = df['PU_avg_market_total_earnings_per_hour'] * (df['average_PU_to_DO_time'] / 3600)

# Convert the relocation opportunity cost to 'opportunity score index units'
converted_relocation_opportunity_cost = relocation_opportunity_cost * conversion_factor
df['converted_relocation_opportunity_cost'] = converted_relocation_opportunity_cost

# Adjust the DO_zone_opportunity_score by subtracting the converted relocation opportunity cost
df['DO_zone_opportunity_score'] = df['DO_zone_opportunity_score'] - df['converted_relocation_opportunity_cost']

print(df[['PUZone', 'DOZone', 'day_of_week', 'hour_bucket', 'average_PU_to_DO_time', 'PU_avg_market_total_earnings_per_hour', 'converted_relocation_opportunity_cost', 'PU_zone_opportunity_score','DO_zone_opportunity_score']].head())
print(f"DataFrame shape after conversion: {df.shape}")

print(df.head())
print(df.shape)

# Check for missing values after the merge and adjustment
print("\nMissing values after merge and adjustment:")
print(df.isnull().sum())

             PUZone                   DOZone day_of_week  hour_bucket  \
0    Yorkville East             East Village    Thursday            0   
1          Red Hook              Cobble Hill    Thursday            0   
2       Boerum Hill               Park Slope    Thursday            0   
3      Midtown East  Greenwich Village North    Thursday            0   
4  Bensonhurst West         Bensonhurst West    Thursday            0   

   average_PU_to_DO_time  PU_avg_market_total_earnings_per_hour  \
0                 889.80                             396.999666   
1                 462.00                             323.168625   
2                 595.50                            1128.520754   
3                 733.65                            1916.840239   
4                   0.00                             344.639696   

   converted_relocation_opportunity_cost  PU_zone_opportunity_score  \
0                             226.362484                1236.554982   
1               

**Reasoning**:
The subtask requires converting the `relocation_opportunity_cost` into 'opportunity score index units' using the previously calculated `conversion_factor`. This step performs that multiplication.



In [13]:
import pandas as pd

# Identify columns with missing 'DO_' values (from previous output, they all have the same count)
missing_do_cols = [
    'DO_zone_opportunity_score',
    'DO_tip_ratio_per_trip',
    'DO_avg_earnings_per_trip',
    'DO_trip_density_per_hour',
    'DO_avg_market_total_earnings_per_hour'
]

# Filter DataFrame to get rows where 'DO_zone_opportunity_score' is null (implies others are also null)
missing_do_rows = df[df['DO_zone_opportunity_score'].isnull()]

print(f"Displaying 10 random rows where 'DO_' prefixed columns are null:")

# Display all columns for the 10 random rows
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(missing_do_rows.sample(10))

Displaying 10 random rows where 'DO_' prefixed columns are null:
            request_datetime   on_scene_datetime     pickup_datetime    dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  tolls   tips  driver_pay  PUBorough                     PUZone DOBorough             DOZone  time_from_request_to_pickup day_of_week  hour_bucket  average_PU_to_DO_time pickup_date  PU_zone_opportunity_score  PU_tip_ratio_per_trip  PU_avg_earnings_per_trip  PU_trip_density_per_hour  PU_avg_market_total_earnings_per_hour  DO_zone_opportunity_score  DO_tip_ratio_per_trip  DO_avg_earnings_per_trip  DO_trip_density_per_hour  DO_avg_market_total_earnings_per_hour  converted_relocation_opportunity_cost
20867281 2026-02-21 21:33:53 2026-02-21 21:36:33 2026-02-21 21:37:48 2026-02-21 22:13:39            62           132       10.17       2152   0.00   0.00       36.25   Brooklyn        Crown Heights South    Queens        JFK Airport                        160.0    Saturday           21      

### Handle Missing DO Zone Metrics

Given that the missing values in the `DO_` prefixed columns primarily correspond to airport zones where we do not want to recommend relocation, we will fill these `NaN` values with `0`. This effectively assigns a zero opportunity score to these drop-off zones, ensuring they are not considered favorable for a driver's next pickup.

In [14]:
# Fill missing values in DO_ prefixed columns with 0
df[missing_do_cols] = df[missing_do_cols].fillna(0)

print("Missing values after filling DO_ prefixed columns with 0:")
print(df[missing_do_cols].isnull().sum())

# Verify a few random rows to ensure the fill operation was successful
print("\n5 random rows from DO_zone_opportunity_score column after filling NaN with 0:")
print(df[['DOZone', 'day_of_week', 'hour_bucket', 'DO_zone_opportunity_score']].sample(5))

Missing values after filling DO_ prefixed columns with 0:
DO_zone_opportunity_score                0
DO_tip_ratio_per_trip                    0
DO_avg_earnings_per_trip                 0
DO_trip_density_per_hour                 0
DO_avg_market_total_earnings_per_hour    0
dtype: int64

5 random rows from DO_zone_opportunity_score column after filling NaN with 0:
                                 DOZone day_of_week  hour_bucket  \
21051743         Charleston/Tottenville      Sunday            8   
10350806                    JFK Airport   Wednesday            2   
14810687      Williamsburg (North Side)    Saturday           12   
14546764                      Homecrest      Friday           21   
16271681  Meatpacking/West Village West     Tuesday           16   

          DO_zone_opportunity_score  
21051743                 199.235161  
10350806                   0.000000  
14810687                9954.684965  
14546764                1437.520547  
16271681                2893.109173 

In [15]:
# Columns to display for a more comprehensive comparison
display_columns = [
    'PUZone',
    'DOZone',
    'day_of_week',
    'hour_bucket',
    'average_PU_to_DO_time',
    'PU_zone_opportunity_score',
    'DO_zone_opportunity_score',
    'PU_avg_market_total_earnings_per_hour',
    'DO_avg_market_total_earnings_per_hour',
    'PU_tip_ratio_per_trip',
    'DO_tip_ratio_per_trip'
]

# 5 random rows where DO opportunity score was higher than PU opportunity score
higher_do_score = df[df['DO_zone_opportunity_score'] > df['PU_zone_opportunity_score']]
print(f"\nNumber of rows where DO opportunity score is HIGHER than PU opportunity score: {higher_do_score.shape[0]}")
print("5 random rows where DO opportunity score is HIGHER than PU opportunity score:")
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(higher_do_score.sample(5)[display_columns])

# 5 random rows where DO opportunity score was lower than PU opportunity score
lower_do_score = df[df['DO_zone_opportunity_score'] < df['PU_zone_opportunity_score']]
print(f"\nNumber of rows where DO opportunity score is LOWER than PU opportunity score: {lower_do_score.shape[0]}")
print("5 random rows where DO opportunity score is LOWER than PU opportunity score:")
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(lower_do_score.sample(5)[display_columns])

# 5 random rows where DO opportunity score was the same as PU opportunity score
same_do_score = df[df['DO_zone_opportunity_score'] == df['PU_zone_opportunity_score']]
print(f"\nNumber of rows where DO opportunity score is the SAME as PU opportunity score: {same_do_score.shape[0]}")
print("5 random rows where DO opportunity score is the SAME as PU opportunity score:")
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(same_do_score.sample(5)[display_columns])


Number of rows where DO opportunity score is HIGHER than PU opportunity score: 8179969
5 random rows where DO opportunity score is HIGHER than PU opportunity score:
                       PUZone                 DOZone day_of_week  hour_bucket  average_PU_to_DO_time  PU_zone_opportunity_score  DO_zone_opportunity_score  PU_avg_market_total_earnings_per_hour  DO_avg_market_total_earnings_per_hour  PU_tip_ratio_per_trip  DO_tip_ratio_per_trip
1823360      Sunset Park East              Bay Ridge     Tuesday            5             501.400000                1032.164640                2268.866331                             354.441488                             708.257686               0.040500               0.090283
15513208        Cypress Hills          East New York      Sunday           19             556.941176                2111.694506                7793.179280                             574.929391                            1712.237332               0.021221               0.0119

## Step 3

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23474050 entries, 0 to 23474049
Data columns (total 31 columns):
 #   Column                                 Dtype         
---  ------                                 -----         
 0   request_datetime                       datetime64[us]
 1   on_scene_datetime                      datetime64[us]
 2   pickup_datetime                        datetime64[us]
 3   dropoff_datetime                       datetime64[us]
 4   PULocationID                           int32         
 5   DOLocationID                           int32         
 6   trip_miles                             float64       
 7   trip_time                              int64         
 8   tolls                                  float64       
 9   tips                                   float64       
 10  driver_pay                             float64       
 11  PUBorough                              object        
 12  PUZone                                 object        


In [17]:
df['day_of_week_numeric'] = df['request_datetime'].dt.dayofweek

columns_to_drop = [
    'request_datetime',
    'on_scene_datetime',
    'pickup_datetime',
    'dropoff_datetime',
    'pickup_date',
    'converted_relocation_opportunity_cost',
    'day_of_week' # Add the original object-type day_of_week to be dropped
]

df = df.drop(columns=columns_to_drop)

print("DataFrame after dropping columns and converting day_of_week:")
print(df.info())
df.head()

DataFrame after dropping columns and converting day_of_week:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23474050 entries, 0 to 23474049
Data columns (total 25 columns):
 #   Column                                 Dtype  
---  ------                                 -----  
 0   PULocationID                           int32  
 1   DOLocationID                           int32  
 2   trip_miles                             float64
 3   trip_time                              int64  
 4   tolls                                  float64
 5   tips                                   float64
 6   driver_pay                             float64
 7   PUBorough                              object 
 8   PUZone                                 object 
 9   DOBorough                              object 
 10  DOZone                                 object 
 11  time_from_request_to_pickup            float64
 12  hour_bucket                            int32  
 13  average_PU_to_DO_time                  

,PULocationID,DOLocationID,trip_miles,trip_time,tolls,tips,driver_pay,PUBorough,PUZone,DOBorough,...,PU_tip_ratio_per_trip,PU_avg_earnings_per_trip,PU_trip_density_per_hour,PU_avg_market_total_earnings_per_hour,DO_zone_opportunity_score,DO_tip_ratio_per_trip,DO_avg_earnings_per_trip,DO_trip_density_per_hour,DO_avg_market_total_earnings_per_hour,day_of_week_numeric
0,262,79,4.30,1133,0.0,0.00,21.10,Manhattan,Yorkville East,Manhattan,...,0.075450,20.799915,21.527813,396.999666,11120.060310,0.055124,21.464324,124.674193,2347.361177,3
1,195,52,1.89,604,0.0,0.00,9.10,Brooklyn,Red Hook,Brooklyn,...,0.040708,25.626590,14.649637,323.168625,570.650786,0.059132,27.429754,11.540584,263.476661,3
2,25,181,1.84,1427,0.0,2.67,21.94,Brooklyn,Boerum Hill,Brooklyn,...,0.061008,25.307699,53.264152,1128.520754,7165.893521,0.052103,23.492268,85.523603,1703.099052,3
3,162,113,2.84,1556,0.0,0.00,20.90,Manhattan,Midtown East,Manhattan,...,0.065770,19.167750,101.123596,1916.840239,2147.058233,0.045348,22.835190,41.113543,814.948261,3
4,22,22,0.66,172,0.0,0.00,4.31,Brooklyn,Bensonhurst West,Brooklyn,...,0.032570,13.702837,27.691697,344.639696,1156.820287,0.032570,13.702837,27.691697,344.639696,3


In [18]:
agg_df = (
    df.groupby([
        'PULocationID',
        'DOLocationID',
        'hour_bucket',
        'day_of_week_numeric'
    ])
    .agg({
        'average_PU_to_DO_time': 'mean',
        'PU_avg_market_total_earnings_per_hour': 'mean',
        'DO_avg_market_total_earnings_per_hour': 'mean',
        'PU_trip_density_per_hour': 'mean',
        'DO_trip_density_per_hour': 'mean',
        'PU_zone_opportunity_score': 'mean',
        'DO_zone_opportunity_score': 'mean'
    })
    .reset_index()
)

print("Aggregated DataFrame Head:")
display(agg_df.head())

Aggregated DataFrame Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score
0,2,10,8,0,805.0,69.227329,859.841769,4.472050,48.371309,117.662456,3317.130280
1,2,10,15,4,1573.0,62.982835,704.553860,2.288620,42.443425,74.979061,2593.711105
2,2,10,16,3,2032.0,62.347841,688.789209,2.532536,43.111691,78.683976,2527.071843
3,2,10,17,2,1811.0,80.062613,604.634029,2.663707,38.049071,103.959339,2122.962694
4,2,10,18,3,1345.0,67.225116,610.723849,2.421796,35.518063,82.698026,2139.326920


In [19]:
agg_df['travel_penalty'] = (
    (agg_df['average_PU_to_DO_time'] / 3600) # Convert seconds to hours
    * agg_df['PU_avg_market_total_earnings_per_hour']
)

print("Aggregated DataFrame with Travel Penalty Head:")
display(agg_df.head())

Aggregated DataFrame with Travel Penalty Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty
0,2,10,8,0,805.0,69.227329,859.841769,4.472050,48.371309,117.662456,3317.130280,15.480000
1,2,10,15,4,1573.0,62.982835,704.553860,2.288620,42.443425,74.979061,2593.711105,27.520000
2,2,10,16,3,2032.0,62.347841,688.789209,2.532536,43.111691,78.683976,2527.071843,35.191893
3,2,10,17,2,1811.0,80.062613,604.634029,2.663707,38.049071,103.959339,2122.962694,40.275942
4,2,10,18,3,1345.0,67.225116,610.723849,2.421796,35.518063,82.698026,2139.326920,25.116050


In [20]:
agg_df['net_gain'] = (
    agg_df['DO_avg_market_total_earnings_per_hour']
    - agg_df['PU_avg_market_total_earnings_per_hour']
    - agg_df['travel_penalty']
)

print("Aggregated DataFrame with Net Gain Head:")
display(agg_df.head())

Aggregated DataFrame with Net Gain Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty,net_gain
0,2,10,8,0,805.0,69.227329,859.841769,4.472050,48.371309,117.662456,3317.130280,15.480000,775.134440
1,2,10,15,4,1573.0,62.982835,704.553860,2.288620,42.443425,74.979061,2593.711105,27.520000,614.051025
2,2,10,16,3,2032.0,62.347841,688.789209,2.532536,43.111691,78.683976,2527.071843,35.191893,591.249474
3,2,10,17,2,1811.0,80.062613,604.634029,2.663707,38.049071,103.959339,2122.962694,40.275942,484.295474
4,2,10,18,3,1345.0,67.225116,610.723849,2.421796,35.518063,82.698026,2139.326920,25.116050,518.382682


In [21]:
stay_rows = (
    agg_df.groupby([
        'PULocationID',
        'hour_bucket',
        'day_of_week_numeric'
    ])
    .first() # Take the first occurrence for each group as a base
    .reset_index()
)

stay_rows['DOLocationID'] = stay_rows['PULocationID']
stay_rows['average_PU_to_DO_time'] = 0
stay_rows['travel_penalty'] = 0

stay_rows['DO_avg_market_total_earnings_per_hour'] = (
    stay_rows['PU_avg_market_total_earnings_per_hour']
)

stay_rows['net_gain'] = 0

print("Stay-Put Rows Head:")
display(stay_rows.head())

Stay-Put Rows Head:


,PULocationID,hour_bucket,day_of_week_numeric,DOLocationID,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty,net_gain
0,2,1,0,2,0,62.481876,62.481876,2.558635,8.792860,79.313066,236.113411,0,0
1,2,6,4,2,0,82.354286,82.354286,6.857143,5.047735,169.767022,299.423956,0,0
2,2,8,0,2,0,69.227329,69.227329,4.472050,48.371309,117.662456,3317.130280,0,0
3,2,8,3,2,0,64.173913,64.173913,4.347826,8.652869,107.599767,557.654012,0,0
4,2,9,2,2,0,105.306733,105.306733,8.977556,18.864806,242.241099,1090.199017,0,0


In [22]:
training_table = pd.concat([
    agg_df,
    stay_rows
], ignore_index=True)

print("Final Training Table Head:")
display(training_table.head())

Final Training Table Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty,net_gain
0,2,10,8,0,805.0,69.227329,859.841769,4.472050,48.371309,117.662456,3317.130280,15.480000,775.134440
1,2,10,15,4,1573.0,62.982835,704.553860,2.288620,42.443425,74.979061,2593.711105,27.520000,614.051025
2,2,10,16,3,2032.0,62.347841,688.789209,2.532536,43.111691,78.683976,2527.071843,35.191893,591.249474
3,2,10,17,2,1811.0,80.062613,604.634029,2.663707,38.049071,103.959339,2122.962694,40.275942,484.295474
4,2,10,18,3,1345.0,67.225116,610.723849,2.421796,35.518063,82.698026,2139.326920,25.116050,518.382682


In [23]:
training_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3185960 entries, 0 to 3185959
Data columns (total 13 columns):
 #   Column                                 Dtype  
---  ------                                 -----  
 0   PULocationID                           int32  
 1   DOLocationID                           int32  
 2   hour_bucket                            int32  
 3   day_of_week_numeric                    int32  
 4   average_PU_to_DO_time                  float64
 5   PU_avg_market_total_earnings_per_hour  float64
 6   DO_avg_market_total_earnings_per_hour  float64
 7   PU_trip_density_per_hour               float64
 8   DO_trip_density_per_hour               float64
 9   PU_zone_opportunity_score              float64
 10  DO_zone_opportunity_score              float64
 11  travel_penalty                         float64
 12  net_gain                               float64
dtypes: float64(9), int32(4)
memory usage: 267.4 MB


## Exporting DataFrames

In [24]:
# Export the main processed training_table 'training_table' to Parquet
output_training_table_path = '/content/uber_trips_training.parquet'
training_table.to_parquet(output_training_table_path, index=False)
print(f"Main DataFrame exported to: {output_training_table_path}")

Main DataFrame exported to: /content/uber_trips_training.parquet


### Export for Zone to Zone Lookups

In [ ]:
# # Export 'average_trip_time_by_route_time' for zone-to-zone lookups
# output_avg_trip_time_path = '/content/avg_trip_time_zone_to_zone.parquet'
# average_trip_time_by_route_time.to_parquet(output_avg_trip_time_path, index=False)
# print(f"Average trip time DataFrame exported to: {output_avg_trip_time_path}")

### Export for Earnings Estimations

In [ ]:
# # Export 'grouped_earnings' for earnings estimations
# output_grouped_earnings_path = '/content/avg_earnings_zone_hour.parquet'
# grouped_earnings.to_parquet(output_grouped_earnings_path, index=False)
# print(f"Grouped earnings DataFrame exported to: {output_grouped_earnings_path}")